## End-to-End ML Pipeline — Complete Checklist

---

### ✅ Step 1: Dataset বোঝা
- Dataset কোথা থেকে এসেছে, কী সমস্যা solve করছে?
- প্রতিটি column-এর meaning বোঝো (data dictionary পড়ো)
- Target column চিহ্নিত করো
- Task কী? → **Classification** (category predict) নাকি **Regression** (number predict)?
- Class imbalance আছে? → `value_counts()` করো

---

### ✅ Step 2: Data Loading ও First Look
- `pd.read_csv()` দিয়ে load করো
- `df.shape`, `df.head()`, `df.dtypes`, `df.info()` দেখো
- `df.describe()` দিয়ে numeric statistics দেখো
- `df.nunique()` দিয়ে unique values দেখো

---

### ✅ Step 3: Missing Values Check
- `df.isnull().sum()` করো — per column null count
- `(df.isnull().mean() * 100)` — percentage missing দেখো
- **Hidden missing** খোঁজো — 0, -1, 999 দিয়ে missing code হয়নি তো?
- Missing > 50% column → drop candidate হিসেবে mark করো

---

### ✅ Step 4: EDA — Distribution দেখো
- Numeric columns → **Histogram** (distribution shape বোঝো)
- Numeric columns → **Boxplot** (outlier চোখে দেখো)
- Categorical columns → **Countplot / Bar chart**
- Target variable distribution দেখো → **Class imbalance** আছে?
- Numeric + Binary features → **Correlation Heatmap** (feature-target সম্পর্ক)

---

### ✅ Step 5: Feature vs Target সম্পর্ক দেখো
- **Numeric feature vs Target** → Boxplot grouped by target (দুটো box আলাদা হলে feature useful)
- **Categorical feature vs Target** → `pd.crosstab(normalize='index')` + Stacked bar chart
- **Correlation** → `df.corr()` দিয়ে target-এর সাথে কোন feature বেশি correlated দেখো
- **Pairplot** → important feature pairs-এর scatter (hue=target)

---

### ✅ Step 6: Train/Test Split করো *(EDA শেষ হলেই split — এর আগে নয়)*
- `train_test_split(X, y, test_size=0.2, random_state=42)`
- Imbalanced হলে → `stratify=y` দাও (দুটো set-এ same proportion রাখতে)
- **এরপর সব preprocessing শুধু train data দিয়ে fit করো**

---

### ✅ Step 7: Missing Values Handle করো *(train data-এর stats দিয়ে)*
- **Numeric + symmetric** → Mean দিয়ে fill
- **Numeric + skewed** → Median দিয়ে fill (outlier-robust)
- **Categorical** → Mode দিয়ে fill
- **Missing > 50%** → Column drop করো
- **Group আছে** → Group mean দিয়ে fill (`groupby().mean()` + `map()`)
- sklearn-এ → `SimpleImputer(strategy='median')` Pipeline-এ রাখো

---

### ✅ Step 8: Outlier Detect ও Handle করো
**Detect:**
- **IQR Method** → Lower = Q1 - 1.5×IQR, Upper = Q3 + 1.5×IQR, এর বাইরে = outlier (non-parametric, distribution assume করে না)
- **Z-Score Method** → |z| > 3 হলে outlier (Normal distribution assume করে)

**Handle:**
- **Remove** → Confirmed data entry error হলে row drop করো
- **Cap (Winsorize)** → Real কিন্তু extreme value হলে `.clip(lower, upper)` করো, sample size বাঁচে
- **Log Transform** → Right-skewed + all positive হলে `np.log(x+1)` করো, range compress হয়

---

### ✅ Step 9: Feature Engineering করো
- **Timestamp আছে** → Hour, day_of_week, is_night, is_weekend extract করো
- **Domain knowledge আছে** → Meaningful feature বানাও (country_mismatch, security_failure_score)
- **Interaction feature** → দুটো feature combine করলে বেশি signal পাওয়া যায় (Age/MaxHR ratio)
- **Binning** → Continuous variable-কে category-তে রূপান্তর করো — `pd.cut()` (interpretability বাড়ে, non-monotonic relationship-এ useful)
- **Polynomial Features** → Non-linear relationship capture করতে `PolynomialFeatures(degree=2)` → Age² বা Age×MaxHR তৈরি হয় (degree বাড়ালে feature explosion + overfitting ঝুঁকি)
- **ID columns** → Drop করো (prediction-এ কাজে লাগে না)

---

### ✅ Step 10: Encoding করো
- **Binary column (2 value)** → Label Encoding (M=1, F=0)
- **Nominal column (কোনো order নেই)** → OneHot Encoding (`get_dummies()` বা `OneHotEncoder`)
- **Ordinal column (order আছে)** → Ordinal Encoding (Low=0, Medium=1, High=2)
- **Dummy Variable Trap** → `drop_first=True` দাও (n categories → n-1 columns যথেষ্ট, multicollinearity avoid করতে)
- **High cardinality** → Target Encoding বা Frequency Encoding consider করো

---

### ✅ Step 11: Scaling করো *(সবসময় train-এ fit, test-এ শুধু transform)*
- **Default choice / outlier আছে** → `StandardScaler` (mean=0, std=1) — formula: z = (x - μ) / σ
- **Fixed range [0,1] দরকার / Neural Network** → `MinMaxScaler` — formula: (x - min) / (max - min)
- **Outlier অনেক বেশি** → `RobustScaler` (IQR দিয়ে scale, outlier-most robust)
- **Tree-based model** (Decision Tree, Random Forest, XGBoost) → Scaling লাগে না (scale-invariant)
- **Data Leakage** → `scaler.fit()` শুধু X_train-এ, test-এ শুধু `.transform()` — `fit_transform(X_test)` কখনো নয়

---

### ✅ Step 12: sklearn Pipeline বানাও
- `ColumnTransformer` দিয়ে numeric ও categorical আলাদা handle করো
- সব preprocessing steps একটি `Pipeline`-এ রাখো
- Data leakage automatically prevent হয়
- একটি object save করলেই নতুন data-তে apply করা যায়

---

### ✅ Step 13: Model Train করো
**Model কোনটা বেছে নেব:**
- **Linear Regression** → Continuous output (number predict), feature-target linear relationship, formula: ŷ = w₁x₁ + w₂x₂ + ... + w₀
- **Logistic Regression** → Binary classification, output = probability via Sigmoid function: σ(z) = 1/(1+e⁻ᶻ), 0.5 threshold-এ class decide করে
- **Decision Tree** → Classification বা Regression, non-linear pattern ধরতে পারে, interpretable
- **Random Forest / XGBoost** → Complex pattern, better accuracy, ensemble method

**Training-এ মনে রাখো:**
- Simple model দিয়ে শুরু করো → Logistic Regression বা Decision Tree (baseline)
- `pipeline.fit(X_train, y_train)` — একটি call-এ সব হয়
- Class imbalanced হলে → `class_weight='balanced'` দাও
- **Cost Function** → Linear Regression-এ MSE minimize করে, Logistic Regression-এ Log Loss minimize করে
- **Gradient Descent** → Cost function-এর gradient দেখে weight iteratively update করে: w = w - α × gradient (α = learning rate)
- Learning rate খুব বড় → diverge করে, খুব ছোট → ধীরে converge করে

---

### ✅ Step 14: Model Evaluate করো
**Classification হলে:**
- **Accuracy** → সঠিক prediction-এর % (imbalanced-এ misleading!)
- **Precision** → Positive predict-এর মধ্যে actually positive কতটুকু (false alarm কম রাখতে)
- **Recall** → Actual positive-এর মধ্যে কতটুকু ধরা গেছে (miss কম রাখতে — medical/fraud-এ critical)
- **F1 Score** → Precision ও Recall-এর harmonic mean (imbalanced-এ best single metric)
- **ROC-AUC** → Different threshold-এ True Positive Rate vs False Positive Rate — 1.0 = perfect, 0.5 = random
- **Confusion Matrix** → TN, FP, FN, TP — FN সবচেয়ে বিপজ্জনক medical/fraud context-এ
- **Decision Boundary** → Default threshold = 0.5, প্রয়োজনে কমিয়ে Recall বাড়ানো যায়

**Regression হলে:**
- **MAE** → Average absolute error, outlier-এ কম sensitive, same unit-এ
- **RMSE** → Root of MSE, large error বেশি penalize করে, outlier-sensitive
- **R² Score** → 0 থেকে 1, "model data-র কতটুকু variance explain করছে" — 1 = perfect, 0 = useless, negative = mean-এর চেয়েও খারাপ

---

### ✅ Step 15: Sanity Check ও Iterate করো
- **Cross-Validation** → `cross_val_score(pipeline, X, y, cv=5)` — mean ± std দেখো, std কম = stable model
- **Training vs Test accuracy gap** → gap < 5% = good generalization, gap বড় = overfitting
- **Overfitting** → Training accuracy >> Test accuracy — model memorize করেছে (fix: more data, regularization, simpler model)
- **Underfitting** → Training accuracy-ও কম — model too simple (fix: complex model, more features)
- **Feature Importance** দেখো → কোন feature বেশি গুরুত্বপূর্ণ, unnecessary feature বাদ দাও
- **Threshold Adjust** করো → Precision-Recall trade-off business requirement অনুযায়ী

---

> **এক কথায় মনে রাখো:**
> `Understand → Load → EDA → Split → Clean → Engineer → Encode → Scale → Pipeline → Train → Evaluate → Iterate`